# Building a device and calibrating lab-frame gates

This notebook follows a calibration workflow: construct one physical two-qubit
`QuantumDevice`, register one complete lab-frame model, tune candidate gates,
independently verify them, and register the accepted calibrations. Frequencies
and Hamiltonian coefficients are in GHz, time is in ns, and propagation uses
the package convention `2πH`.

In [ ]:
using Pkg
project_root =
    isfile(joinpath(pwd(), "Project.toml")) ? pwd() : abspath(joinpath(pwd(), ".."))
if Base.active_project() != joinpath(project_root, "Project.toml") &&
   Base.find_package("QuantumDevices") === nothing
    Pkg.activate(project_root)
end

using CairoMakie
using LinearAlgebra
using Markdown
import QuantumDevices as QD
import QuantumToolbox as QT

CairoMakie.activate!(type = "png")

## Independent propagation and frame helpers

The optimizer propagates the complete laboratory Hamiltonian. We repeat that
propagation independently with QuantumToolbox and apply the known idle-frame
virtual-Z correction before comparing with the logical target.

In [ ]:
function labeled_basis(built, labels)
    hcat([vec(ComplexF64.(Array(built.states[label].data))) for label in labels]...)
end

function virtual_z_correction(labels, frame_frequencies, duration)
    frequencies = Tuple(values(frame_frequencies))
    Diagonal(
        ComplexF64[
            cis(
                π * duration * sum(
                    frequency * (2level - 1) for
                    (frequency, level) in zip(frequencies, label)
                ),
            ) for label in labels
        ],
    )
end

function independent_metrics(gate, objective)
    built = QD.model(gate.modelspec)
    hamiltonian = QD.numerical(built, gate)
    initial = QT.qeye_like(hamiltonian)
    initial = initial isa QT.QuantumObjectEvolution ? initial(nothing, 0.0) : initial
    propagator = QT.sesolve(
        2π * hamiltonian,
        initial,
        [0.0, gate.duration];
        progress_bar = Val(false),
    ).states[end]
    basis = labeled_basis(built, objective.states)
    logical = adjoint(basis) * Matrix(propagator.data) * basis
    if objective.frame_frequencies !== nothing
        logical =
            virtual_z_correction(
                objective.states,
                objective.frame_frequencies,
                gate.duration,
            ) * logical
    end
    fidelity = abs(tr(adjoint(objective.target) * logical))^2 / size(logical, 1)^2
    survival = real(tr(adjoint(logical) * logical)) / size(logical, 1)
    (; fidelity = Float64(real(fidelity)), leakage = max(0.0, 1.0 - survival))
end

function state_populations(gate, initial_label, labels; samples = 401)
    built = QD.model(gate.modelspec)
    times = collect(range(0.0, gate.duration; length = samples))
    initial = QT.QuantumObject(
        vec(ComplexF64.(Array(built.states[initial_label].data)));
        type = QT.Ket(),
        dims = Tuple(built.spec.dimension),
    )
    solution = QT.sesolve(
        2π * QD.numerical(built, gate),
        initial,
        times;
        progress_bar = Val(false),
    )
    populations = Dict(
        label => begin
            reference = vec(ComplexF64.(Array(built.states[label].data)))
            [abs(dot(reference, vec(Array(state.data))))^2 for state in solution.states]
        end for label in labels
    )
    (; times, populations)
end

## 1. Construct the physical device

Both qubits retain their physical frequencies. Four microwave parameters form
I/Q control pairs, two parameters tune the qubit frequencies, and one activates
the exchange interaction. Every control is idle at zero.

In [ ]:
const Q1_FREQUENCY = 4.8
const Q2_FREQUENCY = 5.2
const RESONANCE_FREQUENCY = 5.0
const EXCHANGE = 0.012
const FRAME_FREQUENCIES = (q1 = Q1_FREQUENCY, q2 = Q2_FREQUENCY)

q1 = QD.Component(
    QD.QubitSpec(
        Q1_FREQUENCY;
        fixed_frequency = true,
        metadata = Dict(:physical_frequency_GHz => Q1_FREQUENCY, :time_unit => :ns),
    ),
    :q1,
)
q2 = QD.Component(
    QD.QubitSpec(
        Q2_FREQUENCY;
        fixed_frequency = true,
        metadata = Dict(:physical_frequency_GHz => Q2_FREQUENCY, :time_unit => :ns),
    ),
    :q2,
)

function real_control(name, description)
    QD.DeviceParameter(
        name;
        domain = QD.realdomain(),
        fixed = false,
        default = 0.0,
        description,
    )
end

q1_i = real_control(:q1_i, "Qubit 1 in-phase microwave coefficient in GHz.")
q1_q = real_control(:q1_q, "Qubit 1 quadrature microwave coefficient in GHz.")
q2_i = real_control(:q2_i, "Qubit 2 in-phase microwave coefficient in GHz.")
q2_q = real_control(:q2_q, "Qubit 2 quadrature microwave coefficient in GHz.")
q1_shift = real_control(:q1_shift, "Qubit 1 frequency shift in GHz.")
q2_shift = real_control(:q2_shift, "Qubit 2 frequency shift in GHz.")
exchange = real_control(:exchange, "Activated exchange rate in GHz.")

interactions = (
    :q1_i_drive => QD.param(q1_i) * QD.op(:q1, :x),
    :q1_q_drive => QD.param(q1_q) * QD.op(:q1, :x),
    :q2_i_drive => QD.param(q2_i) * QD.op(:q2, :x),
    :q2_q_drive => QD.param(q2_q) * QD.op(:q2, :x),
    :q1_tuning => QD.param(q1_shift) * QD.op(:q1, :z) / 2,
    :q2_tuning => QD.param(q2_shift) * QD.op(:q2, :z) / 2,
    :exchange =>
        -QD.param(exchange) *
        (QD.op(:q1, :p) * QD.op(:q2, :m) + QD.op(:q1, :m) * QD.op(:q2, :p)),
)

chip = QD.QuantumDevice(:illustrative_chip; components = (q1, q2), interactions)
lab_spec = QD.modelspec(
    chip,
    :two_qubit_lab,
    (:q1, :q2),
    Tuple(first.(interactions));
    dressingspec = QD.DressingSpec(minimum_overlap = 0),
)
QD.register!(chip, lab_spec)

lab_model = QD.model(chip, :two_qubit_lab)
@assert size(lab_model.hamiltonian) == (4, 4)
@assert only(keys(chip.modelspecs)) == :two_qubit_lab
chip

The same registered model is used for every calibration below.

In [ ]:
paths = (
    q1_i = QD.ParamPath(:q1_i),
    q1_q = QD.ParamPath(:q1_q),
    q2_i = QD.ParamPath(:q2_i),
    q2_q = QD.ParamPath(:q2_q),
    q1_shift = QD.ParamPath(:q1_shift),
    q2_shift = QD.ParamPath(:q2_shift),
    exchange = QD.ParamPath(:exchange),
)

logical_states = [(0, 0), (0, 1), (1, 0), (1, 1)]
x1_target = ComplexF64[
    0 0 1 0
    0 0 0 1
    1 0 0 0
    0 1 0 0
]
x2_target = ComplexF64[
    0 1 0 0
    1 0 0 0
    0 0 0 1
    0 0 1 0
]
iswap_target = ComplexF64[
    1 0 0 0
    0 0 im 0
    0 im 0 0
    0 0 0 1
]

x1_objective = QD.UnitaryObjective(
    x1_target;
    states = logical_states,
    frame_frequencies = FRAME_FREQUENCIES,
)
x2_objective = QD.UnitaryObjective(
    x2_target;
    states = logical_states,
    frame_frequencies = FRAME_FREQUENCIES,
)
iswap_objective = QD.UnitaryObjective(
    iswap_target;
    states = logical_states,
    frame_frequencies = FRAME_FREQUENCIES,
)

## 2. Calibrate 10 ns Gaussian X gates

The laboratory drive is a slow exact-zero Gaussian envelope multiplied by the
physical carrier. The Q recipe is initially zero, but retaining it makes the
same candidate gate usable as an I/Q warm start for Piccolo.

In [ ]:
const X_DURATION = 10.0
const X_SIGMA = 2.0

function endpoint_gaussian(time, duration, sigma)
    edge = exp(-0.5 * (duration / (2sigma))^2)
    raw = exp(-0.5 * ((time - duration / 2) / sigma)^2)
    (raw - edge) / (1 - edge)
end

function x_candidate(name, i_path, q_path, frequency)
    QD.GateSpec(
        name,
        lab_spec;
        duration = X_DURATION,
        parameters = (peak_envelope = 0.10, sigma = X_SIGMA),
        controls = Dict(
            i_path => QD.CarrierControl(
                (p, time) ->
                    p.peak_envelope * endpoint_gaussian(time, p.duration, p.sigma);
                frequency,
            ),
            q_path => QD.CarrierControl(
                (p, time) -> 0.0;
                frequency,
                phase = -π / 2,
            ),
        ),
    )
end

x1_candidate = x_candidate(:x_q1, paths.q1_i, paths.q1_q, Q1_FREQUENCY)
x2_candidate = x_candidate(:x_q2, paths.q2_i, paths.q2_q, Q2_FREQUENCY)

x_initial_metrics = (
    q1 = independent_metrics(x1_candidate, x1_objective),
    q2 = independent_metrics(x2_candidate, x2_objective),
)
x_results = (
    q1 = QD.optimize(
        x1_candidate,
        x1_target,
        :peak_envelope => (0.0, 0.20);
        states = logical_states,
        frame_frequencies = FRAME_FREQUENCIES,
        iterations = 80,
    ),
    q2 = QD.optimize(
        x2_candidate,
        x2_target,
        :peak_envelope => (0.0, 0.20);
        states = logical_states,
        frame_frequencies = FRAME_FREQUENCIES,
        iterations = 80,
    ),
)

for (result, objective, i_path) in zip(
    (x_results.q1, x_results.q2),
    (x1_objective, x2_objective),
    (paths.q1_i, paths.q2_i),
)
    verified = independent_metrics(result.gate, objective)
    @assert result.gate.modelspec === lab_spec
    @assert result.gate.duration == X_DURATION
    @assert 0.0 <= result.minimizer[:peak_envelope] <= 0.20
    @assert result.gate.controls[i_path](0.0) == 0.0
    @assert isapprox(result.gate.controls[i_path](X_DURATION), 0.0; atol = 1e-12)
    @assert result.fidelity > 0.99
    @assert isapprox(verified.fidelity, result.fidelity; atol = 1e-8)
    @assert isapprox(verified.leakage, result.leakage; atol = 1e-8)
    QD.register!(chip, result.gate)
end

x_summary = [
    (
        gate = result.gate.name,
        carrier_GHz = frequency,
        initial_fidelity = initial.fidelity,
        optimized_peak_MHz = 1000 * result.minimizer[:peak_envelope],
        fidelity = result.fidelity,
        leakage = result.leakage,
    ) for (result, frequency, initial) in zip(
        (x_results.q1, x_results.q2),
        (Q1_FREQUENCY, Q2_FREQUENCY),
        (x_initial_metrics.q1, x_initial_metrics.q2),
    )
]
x_summary

Plot both the calibrated envelopes and the actual lab-frame microwave
coefficients. The latter visibly resolves the 4.8 and 5.2 GHz carriers.

In [ ]:
x_times = collect(range(0.0, X_DURATION; length = 2001))
x_figure = Figure(size = (820, 560))
envelope_axis = Axis(
    x_figure[1, 1];
    title = "Calibrated Gaussian envelopes",
    xlabel = "time (ns)",
    ylabel = "envelope (MHz)",
)
waveform_axis = Axis(
    x_figure[2, 1];
    title = "Lab-frame microwave coefficients",
    xlabel = "time (ns)",
    ylabel = "coefficient (MHz)",
)
for (result, path, frequency, color) in zip(
    (x_results.q1, x_results.q2),
    (paths.q1_i, paths.q2_i),
    (Q1_FREQUENCY, Q2_FREQUENCY),
    (:dodgerblue, :darkorange),
)
    envelope =
        result.minimizer[:peak_envelope] .*
        [endpoint_gaussian(time, X_DURATION, X_SIGMA) for time in x_times]
    lines!(
        envelope_axis,
        x_times,
        1000 .* envelope;
        label = "$frequency GHz carrier",
        linewidth = 3,
        color,
    )
    lines!(
        waveform_axis,
        x_times,
        1000 .* [result.gate.controls[path](time) for time in x_times];
        linewidth = 1.5,
        color,
    )
end
axislegend(envelope_axis; position = :lt)
x_figure

In [ ]:
x_population = state_populations(chip.gatespecs[:x_q1], (0, 0), [(0, 0), (1, 0)])
x_population_figure = Figure(size = (820, 300))
x_population_axis = Axis(
    x_population_figure[1, 1];
    title = "Registered q1 X calibration: |00⟩ → |10⟩",
    xlabel = "time (ns)",
    ylabel = "population",
)
for (label, color) in zip(((0, 0), (1, 0)), (:dodgerblue, :darkorange))
    lines!(
        x_population_axis,
        x_population.times,
        x_population.populations[label];
        label = "|$(join(label))⟩",
        linewidth = 3,
        color,
    )
end
axislegend(x_population_axis; position = :rc)
x_population_figure

## 3. Calibrate a tunable iSWAP

Starting from 4.8 and 5.2 GHz, cos² ramps tune both qubits toward 5 GHz while
the exchange channel rises to 12 MHz. The optimized plateau offsets represent
residual flux-calibration corrections relative to 5 GHz.

In [ ]:
function cos2_flattop(time, duration, ramp_time)
    time <= 0 && return 0.0
    time >= duration && return 0.0
    time < ramp_time && return sinpi(time / (2ramp_time))^2
    time > duration - ramp_time &&
        return sinpi((duration - time) / (2ramp_time))^2
    1.0
end

iswap_candidate = QD.GateSpec(
    :iswap,
    lab_spec;
    duration = 30.0,
    parameters = (ramp_time = 4.0, q1_plateau_offset = 0.0, q2_plateau_offset = 0.0),
    controls = Dict(
        paths.q1_shift => ((p, time) ->
            (RESONANCE_FREQUENCY + p.q1_plateau_offset - Q1_FREQUENCY) *
            cos2_flattop(time, p.duration, p.ramp_time)),
        paths.q2_shift => ((p, time) ->
            (RESONANCE_FREQUENCY + p.q2_plateau_offset - Q2_FREQUENCY) *
            cos2_flattop(time, p.duration, p.ramp_time)),
        paths.exchange => ((p, time) ->
            EXCHANGE * cos2_flattop(time, p.duration, p.ramp_time)),
    ),
)
iswap_initial = independent_metrics(iswap_candidate, iswap_objective)
iswap_result = QD.optimize(
    iswap_candidate,
    iswap_target,
    (
        :duration => (24.0, 60.0),
        :ramp_time => (2.0, 8.0),
        :q1_plateau_offset => (-0.05, 0.05),
        :q2_plateau_offset => (-0.05, 0.05),
    );
    states = logical_states,
    frame_frequencies = FRAME_FREQUENCIES,
    iterations = 200,
)
iswap_verified = independent_metrics(iswap_result.gate, iswap_objective)

@assert iswap_result.gate.modelspec === lab_spec
@assert 24.0 <= iswap_result.gate.duration <= 60.0
@assert 2.0 <= iswap_result.minimizer[:ramp_time] <= 8.0
@assert -0.05 <= iswap_result.minimizer[:q1_plateau_offset] <= 0.05
@assert -0.05 <= iswap_result.minimizer[:q2_plateau_offset] <= 0.05
@assert iswap_result.fidelity > iswap_initial.fidelity
@assert iswap_result.fidelity > 0.99
@assert isapprox(iswap_verified.fidelity, iswap_result.fidelity; atol = 1e-8)
@assert isapprox(iswap_verified.leakage, iswap_result.leakage; atol = 1e-8)
for path in (paths.q1_shift, paths.q2_shift, paths.exchange)
    @assert iswap_result.gate.controls[path](0.0) == 0.0
    @assert isapprox(
        iswap_result.gate.controls[path](iswap_result.gate.duration),
        0.0;
        atol = 1e-12,
    )
end
QD.register!(chip, iswap_result.gate)

iswap_summary = (
    initial_fidelity = iswap_initial.fidelity,
    duration_ns = iswap_result.gate.duration,
    ramp_ns = iswap_result.minimizer[:ramp_time],
    q1_plateau_GHz =
        RESONANCE_FREQUENCY + iswap_result.minimizer[:q1_plateau_offset],
    q2_plateau_GHz =
        RESONANCE_FREQUENCY + iswap_result.minimizer[:q2_plateau_offset],
    exchange_MHz = 1000 * EXCHANGE,
    fidelity = iswap_result.fidelity,
    leakage = iswap_result.leakage,
)
iswap_summary

In [ ]:
iswap_times = collect(range(0.0, iswap_result.gate.duration; length = 501))
q1_frequencies =
    Q1_FREQUENCY .+
    [iswap_result.gate.controls[paths.q1_shift](time) for time in iswap_times]
q2_frequencies =
    Q2_FREQUENCY .+
    [iswap_result.gate.controls[paths.q2_shift](time) for time in iswap_times]
exchange_rates =
    1000 .*
    [iswap_result.gate.controls[paths.exchange](time) for time in iswap_times]

iswap_pulse_figure = Figure(size = (820, 540))
frequency_axis = Axis(
    iswap_pulse_figure[1, 1];
    title = "Calibrated cos²-ramped iSWAP",
    xlabel = "time (ns)",
    ylabel = "absolute frequency (GHz)",
)
lines!(frequency_axis, iswap_times, q1_frequencies; label = "q1", linewidth = 3)
lines!(frequency_axis, iswap_times, q2_frequencies; label = "q2", linewidth = 3)
axislegend(frequency_axis; position = :rc)
exchange_axis =
    Axis(iswap_pulse_figure[2, 1]; xlabel = "time (ns)", ylabel = "exchange (MHz)")
lines!(exchange_axis, iswap_times, exchange_rates; linewidth = 3, color = :purple)
iswap_pulse_figure

In [ ]:
iswap_population = state_populations(chip.gatespecs[:iswap], (1, 0), [(1, 0), (0, 1)])
iswap_population_figure = Figure(size = (820, 300))
iswap_population_axis = Axis(
    iswap_population_figure[1, 1];
    title = "Registered iSWAP calibration",
    xlabel = "time (ns)",
    ylabel = "population",
)
for (label, color) in zip(((1, 0), (0, 1)), (:dodgerblue, :darkorange))
    lines!(
        iswap_population_axis,
        iswap_population.times,
        iswap_population.populations[label];
        label = "|$(join(label))⟩",
        linewidth = 3,
        color,
    )
end
axislegend(iswap_population_axis; position = :rc)
iswap_population_figure

## 4. Inspect the calibrated device

Registration stores independent snapshots. All three accepted gates therefore
retain the exact model and controls used during calibration.

In [ ]:
@assert Set(keys(chip.modelspecs)) == Set((:two_qubit_lab,))
@assert Set(keys(chip.gatespecs)) == Set((:x_q1, :x_q2, :iswap))
for (name, objective, result) in zip(
    (:x_q1, :x_q2, :iswap),
    (x1_objective, x2_objective, iswap_objective),
    (x_results.q1, x_results.q2, iswap_result),
)
    stored = chip.gatespecs[name]
    @assert stored.modelspec !== lab_spec
    @assert isapprox(
        independent_metrics(stored, objective).fidelity,
        result.fidelity;
        atol = 1e-8,
    )
end
chip

## Advanced: builder-oriented optimization

The concise API is appropriate for calibration recipes. The lower-level API
remains useful when candidate construction cannot be expressed through named
gate parameters.

In [ ]:
advanced_problem = QD.EnvelopeOptimizationProblem(
    values -> QD.with_parameters(x1_candidate; peak_envelope = values.peak_envelope),
    [QD.GateVariable(:peak_envelope; initial = 0.10, lower = 0.0, upper = 0.20)],
    x1_objective,
)

## 5. Optional Piccolo envelope synthesis

Piccolo remains a weak dependency. In a separate environment, develop this
checkout and install Piccolo plus IJulia. `CarrierControl` lets Piccolo optimize
smooth I/Q envelopes while its time-dependent dynamics retain the physical
microwave carriers.

```sh
julia --project=/path/to/piccolo-demo -e \
  'using Pkg; Pkg.develop(path="/path/to/QuantumDevices"); Pkg.add(["Piccolo", "IJulia"])'
```

In [ ]:
piccolo_available = Base.find_package("Piccolo") !== nothing
if piccolo_available
    @eval import Piccolo
    md"Piccolo is available; running independent lab-frame envelope synthesis."
else
    md"Piccolo is not installed in the active environment, so this optional section is skipped."
end

In [ ]:
piccolo_results = if piccolo_available
    function piccolo_x_seed(name, i_path, q_path, frequency)
        QD.GateSpec(
            name,
            lab_spec;
            duration = X_DURATION,
            parameters = NamedTuple(),
            controls = Dict(
                i_path => QD.CarrierControl(
                    time -> 0.02 * sinpi(time / X_DURATION)^2;
                    frequency,
                ),
                q_path => QD.CarrierControl(
                    time -> 0.0;
                    frequency,
                    phase = -π / 2,
                ),
            ),
        )
    end

    x1_seed = piccolo_x_seed(:piccolo_x_q1, paths.q1_i, paths.q1_q, Q1_FREQUENCY)
    x2_seed = piccolo_x_seed(:piccolo_x_q2, paths.q2_i, paths.q2_q, Q2_FREQUENCY)
    piccolo_x1 = QD.optimize(
        x1_seed,
        x1_objective,
        (paths.q1_i => (-0.20, 0.20), paths.q1_q => (-0.20, 0.20));
        backend = :piccolo,
        knots = 31,
        template_kwargs = Dict(:Q => 500.0, :R => 1e-3, :ddu_bound => 0.10),
        solve_kwargs = Dict(:max_iter => 60, :verbose => false, :print_level => 0),
    )
    piccolo_x2 = QD.optimize(
        x2_seed,
        x2_objective,
        (paths.q2_i => (-0.20, 0.20), paths.q2_q => (-0.20, 0.20));
        backend = :piccolo,
        knots = 31,
        template_kwargs = Dict(:Q => 500.0, :R => 1e-3, :ddu_bound => 0.10),
        solve_kwargs = Dict(:max_iter => 60, :verbose => false, :print_level => 0),
    )

    piccolo_iswap_seed = QD.GateSpec(
        :piccolo_iswap,
        lab_spec;
        duration = iswap_result.gate.duration,
        controls = Dict(
            paths.q1_shift => (time ->
                (RESONANCE_FREQUENCY - Q1_FREQUENCY) *
                sinpi(time / iswap_result.gate.duration)^2),
            paths.q2_shift => (time ->
                (RESONANCE_FREQUENCY - Q2_FREQUENCY) *
                sinpi(time / iswap_result.gate.duration)^2),
            paths.exchange => (time ->
                0.5 * EXCHANGE * sinpi(time / iswap_result.gate.duration)^2),
        ),
    )
    piccolo_iswap = QD.optimize(
        piccolo_iswap_seed,
        iswap_objective,
        (
            paths.q1_shift => (0.0, 0.25),
            paths.q2_shift => (-0.25, 0.0),
            paths.exchange => (-EXCHANGE, EXCHANGE),
        );
        backend = :piccolo,
        knots = 41,
        template_kwargs = Dict(:Q => 1000.0, :R => 1e-3, :ddu_bound => 0.10),
        solve_kwargs = Dict(:max_iter => 100, :verbose => false, :print_level => 0),
    )

    for (result, objective) in zip(
        (piccolo_x1, piccolo_x2, piccolo_iswap),
        (x1_objective, x2_objective, iswap_objective),
    )
        verified = independent_metrics(result.gate, objective)
        @assert isfinite(result.fidelity)
        @assert isapprox(verified.fidelity, result.fidelity; atol = 1e-8)
        @assert result.gate.control_recipes === nothing
        QD.register!(chip, result.gate)
    end
    (x1 = piccolo_x1, x2 = piccolo_x2, iswap = piccolo_iswap)
else
    nothing
end

In [ ]:
if piccolo_available
    piccolo_summary = [
        (
            gate = result.gate.name,
            duration_ns = result.gate.duration,
            fidelity = result.fidelity,
            leakage = result.leakage,
            carrier_paths = result.settings[:carrier_paths],
            diagnostic_type = nameof(typeof(result.backend_result)),
        ) for result in (piccolo_results.x1, piccolo_results.x2, piccolo_results.iswap)
    ]
    piccolo_summary
end

In [ ]:
if piccolo_available
    comparison_times = collect(range(0.0, X_DURATION; length = 1001))
    piccolo_comparison = Figure(size = (820, 560))
    envelope_axis = Axis(
        piccolo_comparison[1, 1];
        title = "Classical and Piccolo q1 envelopes",
        xlabel = "time (ns)",
        ylabel = "envelope (MHz)",
    )
    physical_axis = Axis(
        piccolo_comparison[2, 1];
        title = "Piccolo reconstructed lab-frame waveform",
        xlabel = "time (ns)",
        ylabel = "coefficient (MHz)",
    )
    classical_envelope =
        x_results.q1.minimizer[:peak_envelope] .*
        [endpoint_gaussian(time, X_DURATION, X_SIGMA) for time in comparison_times]
    lines!(
        envelope_axis,
        comparison_times,
        1000 .* classical_envelope;
        label = "classical Gaussian",
        linewidth = 3,
    )
    initial_envelopes = piccolo_results.x1.settings[:initial_controls]
    initial_times = collect(range(0.0, X_DURATION; length = size(initial_envelopes, 2)))
    lines!(
        envelope_axis,
        initial_times,
        1000 .* initial_envelopes[1, :];
        label = "Piccolo independent seed",
        linewidth = 2,
    )
    lines!(
        physical_axis,
        comparison_times,
        1000 .* [
            piccolo_results.x1.gate.controls[paths.q1_i](time) for
            time in comparison_times
        ];
        linewidth = 1.5,
        color = :purple,
    )
    axislegend(envelope_axis; position = :lt)
    piccolo_comparison
end

In [ ]:
if piccolo_available
    @assert Set(keys(chip.gatespecs)) == Set((
        :x_q1,
        :x_q2,
        :iswap,
        :piccolo_x_q1,
        :piccolo_x_q2,
        :piccolo_iswap,
    ))
    chip
end

---

*This notebook was generated using [Literate.jl](https://github.com/fredrikekre/Literate.jl).*